In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("/content/hotel_bookings_raw.csv.zip")
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,INFLATION,INFLATION_CHG,CSMR_SENT,UNRATE,INTRSRT,GDP,FUEL_PRCS,CPI_HOTELS,US_GINI,DIS_INC
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,1.8,0.0,93.1,5.3,0.75,18306.96,194.0,0.187566,41.2,41355.0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,1.8,0.0,93.1,5.3,0.75,18306.96,194.0,0.187566,41.2,41355.0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,1.8,0.0,93.1,5.3,0.75,18306.96,194.0,0.187566,41.2,41355.0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,1.8,0.0,93.1,5.3,0.75,18306.96,194.0,0.187566,41.2,41355.0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,1.8,0.0,93.1,5.3,0.75,18306.96,194.0,0.187566,41.2,41355.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 43 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [4]:
# by3od kam 7aga fe kol class (0 = not canceled, 1 = canceled)
# 3shan nfham el distribution beta3 el target w nshoof hal fe imbalance wala la
df["is_canceled"].value_counts()

,count
is_canceled,
0,75166
1,44224


# Data Understanding

In [5]:
df.shape

(119390, 43)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 43 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [7]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date', 'MO_YR', 'CPI_AVG',
       'INFLATION', 'INFLATION_CHG', 'CSMR_SENT', 'UNRATE', 'INTRSRT', 'GDP',
       'FUEL_PRCS', 'CPI_HOTELS', 'US_GINI', 'DIS_INC'],
      dtype='object')

In [8]:
# bn3rd el columns deh 3shan nshoof eh el values beta3tha
df[["reservation_status", "reservation_status_date"]].head()

,reservation_status,reservation_status_date
0,Check-Out,7/1/2015
1,Check-Out,7/1/2015
2,Check-Out,7/2/2015
3,Check-Out,7/2/2015
4,Check-Out,7/3/2015


# Data Cleaning

In [9]:
# bnshof el data leakage columns w bn7zfhom 3shan ma yghlash el model
df = df.drop(["reservation_status", "reservation_status_date"], axis=1)

In [10]:
df[['arrival_date_year',
'arrival_date_month',
'arrival_date_day_of_month']].head()

,arrival_date_year,arrival_date_month,arrival_date_day_of_month
0,2015,July,1
1,2015,July,1
2,2015,July,1
3,2015,July,1
4,2015,July,1


In [11]:
# bn7awl el month mn text le rakam 3shan el model yefhamo
df["arrival_date_month"] = pd.to_datetime(df["arrival_date_month"], format="%B").dt.month

In [12]:
df["arrival_date_month"].head()

,arrival_date_month
0,7
1,7
2,7
3,7
4,7


In [13]:
# bn3od el missing values fe kol column 3shan n3rf henantafha ezay
df.isnull().sum().sort_values(ascending=False)

,0
agent,16340
country,488
CPI_HOTELS,181
US_GINI,181
FUEL_PRCS,181
GDP,181
INTRSRT,181
DIS_INC,181
CSMR_SENT,181
CPI_AVG,181


In [14]:
# bn3ml fill le children 3shan el missing kan 2lel
df["children"].fillna(0, inplace=True)

# bnml2 el country b Unknown 3shan dah column categorical
df["country"].fillna("Unknown", inplace=True)

# bn7zf el agent 3shan fe missing kteer w howa ID msh mohem
df.drop("agent", axis=1, inplace=True)

In [15]:
# bnml2 el missing fe el columns el ra2meya b median 3shan yb2a a7san ma3 el outliers
economic_cols = ["CPI_AVG", "INFLATION", "INFLATION_CHG", "CSMR_SENT",
                 "UNRATE", "INTRSRT", "GDP", "FUEL_PRCS",
                 "CPI_HOTELS", "US_GINI", "DIS_INC"]

for col in economic_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [16]:
# بنتأكد إن مفيش missing values
df.isnull().sum().sort_values(ascending=False).head(10)

,0
hotel,0
is_canceled,0
lead_time,0
arrival_date_year,0
arrival_date_month,0
arrival_date_week_number,0
arrival_date_day_of_month,0
stays_in_weekend_nights,0
stays_in_week_nights,0
adults,0


In [17]:
df.shape


(119390, 40)

## EDA

In [18]:
# bn3od cancelled w not cancelled fe kol no3 hotel
df.groupby("hotel")["is_canceled"].value_counts()

hotel         is_canceled
City Hotel    0              46228
              1              33102
Resort Hotel  0              28938
              1              11122
Name: count, dtype: int64

##City hotels have a higher cancellation rate compared to resort hotels

In [19]:
df['lead_time'].head()

,lead_time
0,342
1,737
2,7
3,13
4,14


In [20]:
df.head(20)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,INFLATION,INFLATION_CHG,CSMR_SENT,UNRATE,INTRSRT,GDP,FUEL_PRCS,CPI_HOTELS,US_GINI,DIS_INC
0,Resort Hotel,0,342,2015,7,27,1,0,0,2,...,1.8,0.0,93.1,5.3,0.75,18306.960,194.0,0.187566,41.2,41355.0
1,Resort Hotel,0,737,2015,7,27,1,0,0,2,...,1.8,0.0,93.1,5.3,0.75,18306.960,194.0,0.187566,41.2,41355.0
2,Resort Hotel,0,7,2015,7,27,1,0,1,1,...,1.8,0.0,93.1,5.3,0.75,18306.960,194.0,0.187566,41.2,41355.0
3,Resort Hotel,0,13,2015,7,27,1,0,1,1,...,1.8,0.0,93.1,5.3,0.75,18306.960,194.0,0.187566,41.2,41355.0
4,Resort Hotel,0,14,2015,7,27,1,0,2,2,...,1.8,0.0,93.1,5.3,0.75,18306.960,194.0,0.187566,41.2,41355.0
5,Resort Hotel,0,14,2015,7,27,1,0,2,2,...,1.8,0.0,93.1,5.3,0.75,18306.960,194.0,0.187566,41.2,41355.0
6,Resort Hotel,0,0,2015,7,27,1,0,2,2,...,1.8,0.0,93.1,5.3,0.75,18306.960,194.0,0.187566,41.2,41355.0
7,Resort Hotel,0,9,2015,7,27,1,0,2,2,...,1.8,0.0,93.1,5.3,0.75,18306.960,194.0,0.187566,41.2,41355.0
8,Resort Hotel,1,85,2015,7,27,1,0,3,2,...,1.7,-0.1,90.7,5.4,0.75,18193.707,202.6,0.185620,41.2,41290.0
9,Resort Hotel,1,75,2015,7,27,1,0,3,2,...,1.8,0.0,95.9,5.4,0.75,18193.707,183.8,0.216699,41.2,41248.0


In [21]:
# bnshof average lead_time le kol group (canceled vs not)
df.groupby("is_canceled")["lead_time"].mean()

,lead_time
is_canceled,
0,79.984687
1,144.848815


# insight: customers with higher lead_time are more likely to cancel

In [22]:
# bnshof kol el columns elly no3ha object (text)
df.select_dtypes(include="object").columns

Index(['hotel', 'meal', 'country', 'market_segment', 'distribution_channel',
       'reserved_room_type', 'assigned_room_type', 'deposit_type',
       'customer_type', 'MO_YR'],
      dtype='object')

In [23]:
df['MO_YR'].head()

,MO_YR
0,7-2015
1,7-2015
2,7-2015
3,7-2015
4,7-2015


In [24]:
# bn7zf MO_YR 3shan howa mكرر ma3 year w month
df.drop("MO_YR", axis=1, inplace=True)

# Preprocessing

In [25]:
# bn7awl hotel le 0 w 1 3shan howa binary column
df["hotel"] = df["hotel"].map({
    "Resort Hotel": 0,
    "City Hotel": 1
})

In [26]:

df["hotel"].head()

,hotel
0,0
1,0
2,0
3,0
4,0


In [27]:
# bn7awl meal le columns keteer (one-hot encoding)
df = pd.get_dummies(df, columns=["meal"], drop_first=True)

In [28]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,INTRSRT,GDP,FUEL_PRCS,CPI_HOTELS,US_GINI,DIS_INC,meal_FB,meal_HB,meal_SC,meal_Undefined
0,0,0,342,2015,7,27,1,0,0,2,...,0.75,18306.96,194.0,0.187566,41.2,41355.0,False,False,False,False
1,0,0,737,2015,7,27,1,0,0,2,...,0.75,18306.96,194.0,0.187566,41.2,41355.0,False,False,False,False
2,0,0,7,2015,7,27,1,0,1,1,...,0.75,18306.96,194.0,0.187566,41.2,41355.0,False,False,False,False
3,0,0,13,2015,7,27,1,0,1,1,...,0.75,18306.96,194.0,0.187566,41.2,41355.0,False,False,False,False
4,0,0,14,2015,7,27,1,0,2,2,...,0.75,18306.96,194.0,0.187566,41.2,41355.0,False,False,False,False


In [29]:
# bn3od el missing fe el columns elly 3ayzen n-encode
cols = ["country", "market_segment", "distribution_channel",
        "reserved_room_type", "assigned_room_type",
        "deposit_type", "customer_type"]

df[cols].isnull().sum()

,0
country,0
market_segment,0
distribution_channel,0
reserved_room_type,0
assigned_room_type,0
deposit_type,0
customer_type,0


In [30]:
# bnshof el unique values fe kol column
df["deposit_type"].unique()
df["customer_type"].unique()
df["market_segment"].unique()
df["country"].unique()

array(['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA', 'Unknown', 'ROU', 'NOR',
       'OMN', 'ARG', 'POL', 'DEU', 'BEL', 'CHE', 'CN', 'GRC', 'ITA',
       'NLD', 'DNK', 'RUS', 'SWE', 'AUS', 'EST', 'CZE', 'BRA', 'FIN',
       'MOZ', 'BWA', 'LUX', 'SVN', 'ALB', 'IND', 'CHN', 'MEX', 'MAR',
       'UKR', 'SMR', 'LVA', 'PRI', 'SRB', 'CHL', 'AUT', 'BLR', 'LTU',
       'TUR', 'ZAF', 'AGO', 'ISR', 'CYM', 'ZMB', 'CPV', 'ZWE', 'DZA',
       'KOR', 'CRI', 'HUN', 'ARE', 'TUN', 'JAM', 'HRV', 'HKG', 'IRN',
       'GEO', 'AND', 'GIB', 'URY', 'JEY', 'CAF', 'CYP', 'COL', 'GGY',
       'KWT', 'NGA', 'MDV', 'VEN', 'SVK', 'FJI', 'KAZ', 'PAK', 'IDN',
       'LBN', 'PHL', 'SEN', 'SYC', 'AZE', 'BHR', 'NZL', 'THA', 'DOM',
       'MKD', 'MYS', 'ARM', 'JPN', 'LKA', 'CUB', 'CMR', 'BIH', 'MUS',
       'COM', 'SUR', 'UGA', 'BGR', 'CIV', 'JOR', 'SYR', 'SGP', 'BDI',
       'SAU', 'VNM', 'PLW', 'QAT', 'EGY', 'PER', 'MLT', 'MWI', 'ECU',
       'MDG', 'ISL', 'UZB', 'NPL', 'BHS', 'MAC', 'TGO', 'TWN', 'DJI',
       'STP', 'KN

In [31]:
df["deposit_type"].head()

,deposit_type
0,No Deposit
1,No Deposit
2,No Deposit
3,No Deposit
4,No Deposit


In [32]:
df['customer_type'].head()

,customer_type
0,Transient
1,Transient
2,Transient
3,Transient
4,Transient


In [33]:
# bn7awl deposit_type le arqam (one-hot encoding)
df = pd.get_dummies(df, columns=["deposit_type"], drop_first=True)

In [34]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,FUEL_PRCS,CPI_HOTELS,US_GINI,DIS_INC,meal_FB,meal_HB,meal_SC,meal_Undefined,deposit_type_Non Refund,deposit_type_Refundable
0,0,0,342,2015,7,27,1,0,0,2,...,194.0,0.187566,41.2,41355.0,False,False,False,False,False,False
1,0,0,737,2015,7,27,1,0,0,2,...,194.0,0.187566,41.2,41355.0,False,False,False,False,False,False
2,0,0,7,2015,7,27,1,0,1,1,...,194.0,0.187566,41.2,41355.0,False,False,False,False,False,False
3,0,0,13,2015,7,27,1,0,1,1,...,194.0,0.187566,41.2,41355.0,False,False,False,False,False,False
4,0,0,14,2015,7,27,1,0,2,2,...,194.0,0.187566,41.2,41355.0,False,False,False,False,False,False


In [35]:
cat_cols = ["country", "market_segment", "distribution_channel",
            "reserved_room_type", "assigned_room_type",
            "customer_type"]

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [36]:
df.shape

(119390, 248)

# Splitting Data

In [37]:
x = df.drop("is_canceled", axis=1)
y = df["is_canceled"]

In [38]:
from sklearn.model_selection import train_test_split

# bn2asem el data le training w testing
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [39]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((95512, 247), (23878, 247), (95512,), (23878,))

# Model Building

In [40]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [41]:
# bn3ml prediction 3la el test data
y_pred = model.predict(X_test)
# bn7sb accuracy
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.6877879219365106

In [42]:
# bn3rd report kamel 3an el model
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.87      0.78     14907
           1       0.64      0.39      0.48      8971

    accuracy                           0.69     23878
   macro avg       0.67      0.63      0.63     23878
weighted avg       0.68      0.69      0.67     23878



In [43]:
#  decision tree model
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [44]:
# bn3ml prediction b decision tree
dt_pred = dt_model.predict(X_test)


In [45]:
# bn7sb accuracy bta3 decision tree
accuracy_score(y_test, dt_pred)

0.9230253790099673

In [46]:
# accuracy على training data
dt_model.score(X_train, y_train)

0.9974976966245079

In [47]:
# bn2ll t3eed el shagara 3shan ntfaada el overfitting
dt_model_tuned = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

dt_model_tuned.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=10, min_samples_leaf=10, min_samples_split=20,
                       random_state=42)

In [48]:
# bn3ml prediction b el model el mtzbt
dt_tuned_pred = dt_model_tuned.predict(X_test)

In [49]:
# bn7sb accuracy bta3 el tuned model
accuracy_score(y_test, dt_tuned_pred)

0.866194823687076

In [50]:
# bnshof train accuracy 3shan nt2ked en el overfitting 2all
dt_model_tuned.score(X_train, y_train)

0.8673360415445179

# Random Forest

In [51]:

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [52]:

rf_pred = rf_model.predict(X_test)
accuracy_score(y_test, rf_pred)

0.9508334031325907

# KNN Model

In [53]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier()
knn_model.fit(X_train, y_train)

KNeighborsClassifier()

In [54]:
Knn_pred = knn_model.predict(X_test)
accuracy_score(y_test, Knn_pred)

0.8861294915822095

# Final Model Comparison

In [56]:

print("Logistic Regression Accuracy: ~0.68")
print("Decision Tree Accuracy: ~0.92 (overfitting)")
print("Tuned Decision Tree Accuracy: ~0.86")
print("KNN Accuracy: ~0.88")
print("Random Forest Accuracy: ~0.95 (best)")

Logistic Regression Accuracy: ~0.68
Decision Tree Accuracy: ~0.92 (overfitting)
Tuned Decision Tree Accuracy: ~0.86
KNN Accuracy: ~0.88
Random Forest Accuracy: ~0.95 (best)


# Feature Importance

In [60]:
import pandas as pd

importances = rf_model.feature_importances_

feature_importance = pd.DataFrame({
    "feature": x.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance.head(10)

,feature,importance
34,deposit_type_Non Refund,0.087617
1,lead_time,0.082389
4,arrival_date_week_number,0.079662
3,arrival_date_month,0.061242
170,country_PRT,0.057590
16,adr,0.052477
18,total_of_special_requests,0.042604
5,arrival_date_day_of_month,0.034492
29,DIS_INC,0.025831
7,stays_in_week_nights,0.025029


# Conclusion
# Random Forest achieved the best performance (~95% accuracy).
# Customers with high lead_time are more likely to cancel.
# City hotels have higher cancellation rates.
# Data preprocessing improved model performance significantly.